In [ ]:
import argparse
import json
import concurrent.futures
import re
import os
import sys
from tqdm import tqdm
from PIL import Image
from openai import OpenAI
import multiprocessing
import random
import base64


MODEL_NAME = "logicsparingv2"
INPUT_FILE = "MPDocBench.json"
OUTPUT_PATH = f"./markdown/{MODEL_NAME}"

MD_PATH = OUTPUT_PATH + "_md"
os.makedirs(MD_PATH, exist_ok=True)

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
    raw_data = {}
    for item in data:
        page_info = item["page_info"]
        images_list = page_info["images_list"]
        annotations_list = page_info["annotations_list"]
        image_path = page_info["image_path"]
        pdf_name = os.path.splitext(image_path)[0]
        if pdf_name not in raw_data:
            raw_data[pdf_name] = []
            page_id = 0
            for img, ann in zip(images_list, annotations_list):
                raw_data[pdf_name].append((page_id, img, ann))
                page_id += 1
        else:
            print(f"Warning: duplicate pdf_name {pdf_name} found. Skipping.")
            continue
    data = raw_data

for pdf_name in data:
    orig_mmd_path = os.path.join(OUTPUT_PATH, pdf_name + "_.md")
    md_cnontent = open(orig_mmd_path).read()
    with open(os.path.join(MD_PATH, pdf_name + ".md"), "w") as f:
        f.write(md_cnontent)